## Dataset Simulation using GenAI

API & Configuration Settings

In [4]:
# configure api
from dotenv import load_dotenv
import os

load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")

In [5]:
# Prompt 
from google import genai
from google.genai import types

client = genai.Client(api_key=gemini_api_key)

model = ["gemini-2.0-flash"]

generate_content_config = types.GenerateContentConfig(
    response_mime_type="application/json",
)

Prompt to GenAI

In [ ]:
import time

def generate_patient_batch(batch_size=2, days=30, start_id=20000):

    prompt = f"""You are a healthcare data generator. Generate JSON file containing synthetic patient records for EXACTLY {batch_size * days} records.
                 Each of the {batch_size} patients should have {days} daily entries.

         - patient id (5-digit integer, starting from {start_id})
         - date (YYYY-MM-DD, each {start_id} spanning 30 days)
         - time (HH:MM:SS in 24-hour format)
         - oxygen saturation (95-100% or None for 10% missing)
         - heart rate (60-100 bpm or None for 5% missing)
         - body temperature (36.5-37.5°C or None for 2% missing)
         - blood pressure (100-140 or None for 8% missing)
         - weight (in kilograms or None for 1% missing)
         - blood glucose (70-120 mg/dL or None for 15% missing)
         - Clinical notes or questionnaire responses (related to healthcare responses, more than 20 unique responses)

         Example format:
         [
           "patient_id": 10000,
           "date": "2024-01-01",
           "time": "08:43:53",
           "oxygen_saturation": 97,
           "heart_rate": 70,
           "body_temperature": 36.8,
           "blood_pressure": 110,
           "weight": 70.0,
           "blood_glucose": 90,
           "clinical_notes": "Patient feeling well"
         ],
         [
           "patient_id": 10000,
           "date": "2024-01-02",
           "time": "10:50:35",
           "oxygen_saturation": 98,
           "heart_rate": 72,
           "body_temperature": 37.0,
           "blood_pressure": 115,
           "weight": 70.2,
           "blood_glucose": 92,
           "clinical_notes": "Slight headache"
         ],
         [
           "patient_id": 10000,
           "date": "2024-01-03",
           "time": "16:15:20",
           "oxygen_saturation": 99,
           "heart_rate": 75,
           "body_temperature": 36.9,
           "blood_pressure": 120,
           "weight": 70.5,
           "blood_glucose": 95,
           "clinical_notes": "No issues reported"
         ]
    """

    for attempt in range(3):  # Retry up to 3 times
        try:
            response = client.models.generate_content(model=model[0], contents=prompt, config=generate_content_config)
            return response
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {str(e)}")
            time.sleep(2)
    raise ValueError("Failed to generate valid data after 3 attempts")

# Generate dataset in batches to avoid API limits
def generate_full_dataset(total_patients=500, days=30, batch_size=2):
    for i in range(0, total_patients, batch_size):
        current_batch_size = min(batch_size, total_patients - i)
        start_id = 10000 + i*2  # Ensure unique patient IDs across batches

        print(f"Generating batch {i//batch_size + 1} ({current_batch_size} patients)...")

        batch_data = generate_patient_batch(current_batch_size, days, start_id)
            
      # Save the response as a JSON file
        with open(f"data/{start_id}.json", "w") as file:
          file.write(batch_data.text)

        print("Waiting for 1 minute before the next iteration...")
        time.sleep(60)

generate_full_dataset()

Generating batch 1 (2 patients)...
Waiting for 1 minute before the next iteration...


In [9]:
import csv
from glob import glob
import re

def clean_json_content(content):
    # Handle missing None values
    # Replace unquoted None with quoted "None"
    content = re.sub(r':\s*None\s*([,}])', r': null\1', content, flags=re.IGNORECASE)
    return content

def load_json_file(file_path):
    # Load JSON file with robust error handling
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            if not content:
                return None
                
            # Clean the content
            cleaned_content = clean_json_content(content)
            
            # parse the JSON
            return json.loads(cleaned_content)
                
    except Exception as e:
        print(f"Error reading {file_path}: {str(e)}")
        return None

def combine_json_to_csv(input_dir="data", output_file="patient_data.csv"):
    json_files = glob(os.path.join(input_dir, "*.json"))
    combined_data = []
    
    for file_path in json_files:
        data = load_json_file(file_path)
        if data is None:
            continue
            
        if isinstance(data, list):
            combined_data.extend(data)
        else:
            combined_data.append(data)
    
    if not combined_data:
        print("No valid data found in JSON files")
        return
    
    # Get all unique field names
    fieldnames = set()
    for record in combined_data:
        if isinstance(record, dict):
            fieldnames.update(record.keys())
    
    # Write to CSV with proper None handling
    with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=sorted(fieldnames), 
                              restval='', extrasaction='ignore')
        writer.writeheader()
        
        for record in combined_data:
            if isinstance(record, dict):
                # Convert None values to empty strings for CSV
                cleaned_record = {k: v if v is not None else '' for k, v in record.items()}
                writer.writerow(cleaned_record)
    
    print(f"Successfully processed {len(combined_data)} records to {output_file}")

# Run the function
combine_json_to_csv()

Successfully processed 15000 records to patient_data.csv


## Exploratory Data Analysis (EDA) enhanced by LLMs 

API to HuggingFace

In [ ]:
# Login through terminal huggingface-cli login

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.1")
pipe(messages)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

ValueError: Could not load model mistralai/Mistral-7B-Instruct-v0.1 with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForCausalLM'>, <class 'transformers.models.mistral.modeling_mistral.MistralForCausalLM'>). See the original errors:

while loading with AutoModelForCausalLM, an error is thrown:
Traceback (most recent call last):
  File "d:\Program\Python313\Lib\site-packages\transformers\pipelines\base.py", line 291, in infer_framework_load_model
    model = model_class.from_pretrained(model, **kwargs)
  File "d:\Program\Python313\Lib\site-packages\transformers\models\auto\auto_factory.py", line 573, in from_pretrained
    return model_class.from_pretrained(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        pretrained_model_name_or_path, *model_args, config=config, **hub_kwargs, **kwargs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "d:\Program\Python313\Lib\site-packages\transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_4bit=True)
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1", quantization_config=quantization_config)

In [ ]:
import torch
import pandas as pd

def generate_prompt(prompt):
  # Move model to GPU
  device = "cuda" if torch.cuda.is_available() else "cpu"
  model.to(device)

  # Tokenize input and move to same device as the model
  inputs = tokenizer(prompt, return_tensors="pt").to(device)

  # Generate response
  outputs = model.generate(**inputs, max_length=5000, pad_token_id=tokenizer.eos_token_id)

  print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Distribution of numerical variables

In [ ]:
# import visualisation packages
from matplotlib import pyplot as plt
from scipy import stats
import seaborn as sns
import numpy as np
import plotly.express as px
%matplotlib inline

In [ ]:
# Histogram
plt.figure(figsize=(15, 10))
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(3, 3, i)
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.tight_layout()

plt.show()

In [ ]:
stats_df = pd.DataFrame(index=numeric_cols, columns=[
        'mean', 'median', 'std', 'skew', 'kurtosis', 'min', 'max'])

for col in numeric_cols:
  stats_df.loc[col] = {
            'mean': df[col].mean(),
            'median': df[col].median(),
            'std': df[col].std(),
            'skew': df[col].skew(),
            'kurtosis': df[col].kurtosis(),
            'min': df[col].min(),
            'max': df[col].max()
        }

prompt=f"""Interpret a dataset with these numeric variables: {', '.join(numeric_cols)}.

        Here are the statistical properties:
        {stats_df.to_markdown()}

        Please interpret these distributions by addressing:
        1. Shape of each distribution (normal, skewed, bimodal etc.)
        2. Notable patterns/anomalies
        3. What the KDE reveals about density
        4. Presence of potential outliers
        5. Suggestions for data preprocessing
        6. Comparison between variables

        Provide both technical and business-level interpretations."""

generate_prompt(prompt)